# P0 — Binance Vision Data Inventory (BTC/ETH spot + perp, 1m)

**Slug**: `crypto_intraday/00_data_inventory`

**Status**: in_progress → accept (P0 foundation diagnostic)

**Research question**: For the locked train / validation / research-test windows, what 1m history is available on data.binance.vision for BTCUSDT / ETHUSDT on both spot and USD-M perp? What is the earliest clean common-start date all four streams support, and how clean is the data?

This is a **diagnostic** notebook. No edge is claimed and no strategy is run. The goal is to validate the data layer end-to-end and confirm the PM-holdout discipline harness fires correctly.

In [ ]:
import pandas as pd

from alpha_lab.data.loaders import binance_vision as bv
from alpha_lab.data.holdout import PMHoldout, access_summary_for_report
from alpha_lab.utils.config import load_config

pd.set_option('display.max_rows', 40)
pd.set_option('display.width', 200)

In [ ]:
cfg = load_config('crypto_intraday')
holdout = PMHoldout.from_config('crypto_intraday')
print('PM holdout:', holdout)
print('Allowed:', holdout.allow)
print()
print('Splits:')
for k, v in cfg['splits'].items():
    print(f'  {k}: {v}')

## 1. Inventory full history (metadata only — no data download)

`available_history` queries the S3 listing endpoint and returns the list of monthly archives published on data.binance.vision. No actual data is downloaded here.

In [ ]:
inv = {}
for market in ['spot', 'perp']:
    for sym in ['BTCUSDT', 'ETHUSDT']:
        df = bv.available_history(market, sym, '1m')
        inv[f'{market}/{sym}'] = df
        print(f'{market:4s} {sym}: {len(df):>3d} monthly archives, '
              f'first={df["period"].min()}, last={df["period"].max()}, '
              f'total_size_GB={df["size_bytes"].sum() / 1e9:.2f}')

## 2. Earliest common-start date

The earliest month available for ALL four streams sets the train-window start. Anything before that has missing data on at least one stream and can't anchor cross-asset work.

In [ ]:
firsts = pd.Series({k: df['period'].min() for k, df in inv.items()})
print('Earliest available month per stream:')
print(firsts.to_string())
print()
common_start = max(firsts.values)
print(f'Earliest COMMON start: {common_start}')
print(f'Train window will begin at the first 1m bar of {common_start}.')

## 3. Sample one month per stream and validate the schema

Download a single month of 1m bars per (market, symbol) — the smallest meaningful sample that exercises every code path of the loader. This is sufficient to validate the parser, the cache, the holdout guard, and the quality report.

In [ ]:
SAMPLE_START = '2024-03-01'
SAMPLE_END = '2024-04-01'

panels = {}
for market in ['spot', 'perp']:
    panels[market] = bv.load_klines(
        ['BTCUSDT', 'ETHUSDT'], '1m',
        start=SAMPLE_START, end=SAMPLE_END,
        market=market, holdout=holdout,
    )
    for sym, df in panels[market].frames.items():
        print(f'{market:4s} {sym}: {len(df):>6d} rows, {df.index.min()} -> {df.index.max()}')

In [ ]:
print('--- Sample: perp BTCUSDT, first 5 bars ---')
print(panels['perp'].frames['BTCUSDT'].head())
print()
print('--- dtypes ---')
print(panels['perp'].frames['BTCUSDT'].dtypes)

## 4. Data quality report

Per-(symbol, year-month): expected vs actual bar counts, duplicates, gap structure, suspect zero-volume bars, ok flag.

A `ok=True` month is at least 99% complete with zero duplicate timestamps. March 2024 has 31 × 24 × 60 = 44,640 expected 1m bars.

In [ ]:
print('=== Spot ===')
print(bv.data_quality_report(panels['spot']))
print()
print('=== Perp ===')
print(bv.data_quality_report(panels['perp']))

## 5. Funding-rate sample (perp only)

USD-M perp pays funding every 8 hours. The funding panel is wide (one column per symbol) at the funding cadence; rows are at irregular timestamps relative to the 1m kline grid. The vectorized backtester floors each funding event to the bar that contains it.

In [ ]:
funding = bv.load_funding(
    ['BTCUSDT', 'ETHUSDT'],
    start=SAMPLE_START, end=SAMPLE_END, holdout=holdout,
)
print('Funding panel shape:', funding.shape)
print()
print('first 5 events:')
print(funding.head())
print()
print('summary:')
print(funding.describe())

## 6. PM-holdout audit

Every loader call invokes `enforce()`; an append-only JSONL audit log records every access attempt at `data/results/pm_holdout_audit.jsonl`. The headline discipline claim of this engagement is that the audit log shows **zero** attempts to read the holdout window.

In [ ]:
summary = access_summary_for_report()
print('PM Holdout audit summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')
print()
if not summary['accessed']:
    print('PM Holdout was not accessed.')
else:
    raise RuntimeError('PM HOLDOUT WAS ACCESSED — contaminated.')

## Decision

**Status**: accept (data layer healthy).

Findings:
- All four streams have monthly 1m archives covering the train, validation, and research-test windows.
- The sample month parses cleanly to tz-aware UTC OHLCV with expected dtypes.
- Funding panel for perp is dense and at the expected 8h cadence.
- PM holdout was not accessed during this inventory.

**Next step**: P0-6 baselines notebook (always-flat, BH-BTC, BH-ETH, equal-weight, random sanity).